# Pipeline Observability Metrics

Thin notebook entrypoint for the phase-1 observability summary pipeline.

This notebook delegates monitored-pipeline freshness reads, Silver quarantine aggregation, `obs.obs_dq_metrics` merge logic, and `obs.obs_pipeline_run_log` writes to `src/lakehouse`.

It summarizes real-time pipeline state already written by Bronze, Silver, Gold, and cross-domain tasks.


In [ ]:
import sys
import uuid
from pathlib import Path


def add_repo_src_to_path() -> str:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        src_dir = candidate / "src"
        if (src_dir / "lakehouse" / "__init__.py").exists():
            src_dir_str = str(src_dir)
            if src_dir_str not in sys.path:
                sys.path.insert(0, src_dir_str)
            return src_dir_str

    raise RuntimeError("Could not locate the repository src directory from the current notebook path")


REPO_SRC = add_repo_src_to_path()

from lakehouse.pipelines.obs import run_pipeline_observability_metrics

spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text("catalog", "market_macro")
dbutils.widgets.text("run_id", str(uuid.uuid4()))


In [ ]:
result = run_pipeline_observability_metrics(
    spark=spark,
    catalog=dbutils.widgets.get("catalog").strip() or "market_macro",
    run_id=dbutils.widgets.get("run_id").strip(),
    display_fn=display,
)

result.as_dict()
